In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAÇÕES DE CAMINHOS
# ==========================================
caminho_modelo = './modelo_subm3'
ficheiro_entrada = 'subm3.csv' 
ficheiro_saida = './subm3-g7-MEI-B.csv'
ID2LABEL = {0: 'Human', 1: 'OpenAI', 2: 'Google', 3: 'Meta', 4: 'Anthropic'}

# ==========================================
# 2. CARREGAR MODELO E TOKENIZER
# ==========================================
print("A carregar o melhor modelo...")
tokenizer = AutoTokenizer.from_pretrained(caminho_modelo)
model = AutoModelForSequenceClassification.from_pretrained(caminho_modelo)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# ==========================================
# 3. LER O CSV (COM DELIMITADOR ;)
# ==========================================
# Lemos usando sep=';' para aceitar o teu formato
df = pd.read_csv(ficheiro_entrada, sep=';', encoding='utf-8')

# ==========================================
# 4. FUNÇÃO DE PREVISÃO
# ==========================================
def prever_label(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return 0 
    
    inputs = tokenizer(
        str(texto), 
        return_tensors="pt", 
        truncation=True, 
        max_length=512, 
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Converter para probabilidades (0 a 1)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    
    # Se a probabilidade da classe 1 (OpenAI) for muito próxima da 0 (Humano),
    # podes forçar um critério de desempate se achares que há erro a mais.
    # Por agora, vamos apenas garantir que pegamos na maior com precisão:
    pred_id = torch.argmax(probs).item()
    return ID2LABEL[pred_id]

# ==========================================
# 5. PROCESSAMENTO
# ==========================================
print(f"A processar {len(df)} entradas...")
tqdm.pandas()

# Criamos a nova coluna Label
df['Label'] = df['Text'].progress_apply(prever_label)

# ==========================================
# 6. EXPORTAR APENAS ID E LABEL (DELIMITADOR ;)
# ==========================================
# Selecionamos apenas as colunas pedidas
df_final = df[['ID', 'Label']]

# Guardamos com o delimitador ; e sem o índice do pandas
df_final.to_csv(ficheiro_saida, sep=';', index=False)

print(f"\nConcluído! Ficheiro guardado como: {ficheiro_saida}")
print(df_final.head()) # Mostra as primeiras linhas para conferires

A carregar o melhor modelo...


Loading weights: 100%|██████████| 138/138 [00:00<00:00, 5421.64it/s]


A processar 150 entradas...


100%|██████████| 150/150 [00:04<00:00, 31.39it/s]


Concluído! Ficheiro guardado como: ./subm3-g7-MEI-B.csv
       ID   Label
0  D2-126  OpenAI
1  D2-127  OpenAI
2  D2-128  OpenAI
3  D2-129  OpenAI
4  D2-130  OpenAI
